<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/torneos/notebooks/c6_l4.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C6-L4 · Kelly y stake
Half-Kelly simulado en 60 rondas.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/torneos/data/c6_l4.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c6_l4.csv'), Path('data/c6_l4.csv'), Path('c6_l4.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))


In [ ]:
# De filo y ruido a fracción: Kelly y half-Kelly por ronda
df['f_kelly'] = df['corr_estimado'] / (df['vol'] ** 2)
df['f_half'] = (df['f_kelly'] / 2).clip(lower=0, upper=0.25)
df.loc[df['corr_estimado'] <= 0, ['f_kelly', 'f_half']] = 0.0
print(df[['ronda','corr_estimado','vol','f_kelly','f_half']].head(8).to_string(index=False))
print('Stake medio half-Kelly:', round(df['f_half'].mean(), 4))
assert ((df['f_half'] >= 0) & (df['f_half'] <= 0.25)).all()
assert (df.loc[df['corr_estimado'] <= 0, 'f_half'] == 0).all(), 'sin filo no hay stake'


In [ ]:
# Simulación 60 rondas: fijo vs Kelly full (cap 0.5) vs half-Kelly
rng = np.random.RandomState(7)
ruido = rng.randn(len(df)) * df['vol'].values * 2.0  # ruido alto: apalancarse de más duele
ret = df['corr_estimado'].values * 1.0 + ruido
eq_fijo, eq_full, eq_half = 1.0, 1.0, 1.0
for i, r in df.iterrows():
    k = i - df.index[0]
    eq_fijo *= (1 + 0.10 * ret[k])
    eq_full *= (1 + min(r['f_kelly'], 0.5) * ret[k])
    eq_half *= (1 + r['f_half'] * ret[k])
print(f'Capital final: fijo={eq_fijo:.3f} kelly_full={eq_full:.3f} half_kelly={eq_half:.3f}')
assert all(np.isfinite([eq_fijo, eq_full, eq_half])) and min(eq_fijo, eq_full, eq_half) > 0
assert eq_half > eq_fijo > eq_full, 'half-Kelly >= fijo > full-Kelly en esta demo'


In [ ]:
# Pasarse del óptimo destruye: Kelly x2
eq_doble = 1.0
for i, r in df.iterrows():
    k = i - df.index[0]
    eq_doble *= (1 + min(r['f_kelly'] * 2, 1.0) * ret[k])
print(f'Kelly x2 (sobre-apuesta)={eq_doble:.3f} vs half={eq_half:.3f}')
assert np.isfinite(eq_doble)


In [ ]:
# Chequeo automático L4
assert ((df['f_half'] >= 0) & (df['f_half'] <= 0.25)).all()
assert np.isfinite(eq_half) and eq_half > 0
print('OK L4: Kelly / half-Kelly simulados y verificados')
